# Pessoa 2 - Transfer Learning e comparação final

Este notebook resume a etapa de Transfer Learning do trabalho. Ele não retreina os modelos; apenas reúne os resultados já gerados pelos scripts em `src/` e organiza os artefatos para análise, relatório e apresentação.

Modelos comparados:

- CNN simples treinada do zero pela Pessoa 1;
- ResNet18 com Transfer Learning;
- DenseNet121 com Transfer Learning, como experimento adicional.

## Configuração geral

A ResNet18 e a DenseNet121 foram usadas com pesos pré-treinados no ImageNet. Em ambos os casos, o backbone foi congelado e apenas a camada final foi treinada para classificar as imagens em `NORMAL` e `PNEUMONIA`.

A comparação final usa o conjunto de teste original do dataset. O conjunto de validação possui apenas 16 imagens, então suas métricas devem ser interpretadas com cautela.

## Tabela comparativa final

| Modelo | Acurácia | Precisão | Recall | F1-score |
|---|---:|---:|---:|---:|
| CNN simples | 0.8766 | 0.9003 | 0.9026 | 0.9014 |
| ResNet18 Transfer Learning | 0.9054 | 0.9046 | 0.9487 | 0.9262 |
| DenseNet121 Transfer Learning | 0.8734 | 0.8676 | 0.9410 | 0.9028 |

A ResNet18 apresentou o melhor desempenho geral no conjunto de teste, com maior acurácia, precisão, recall e F1-score entre os modelos comparados.

## Matrizes de confusão

### CNN simples

![Matriz de confusão da CNN simples](../results/notebook_modelo/simple_cnn_confusion_matrix.png)

### ResNet18 Transfer Learning

![Matriz de confusão da ResNet18](../results/transfer_frozen_modelo/transfer_confusion_matrix.png)

### DenseNet121 Transfer Learning

![Matriz de confusão da DenseNet121](../results/transfer_densenet121_modelo/transfer_confusion_matrix.png)

## Histórico de treinamento

### ResNet18 - perda

![Histórico de perda da ResNet18](../results/transfer_frozen_modelo/transfer_loss_history.png)

### ResNet18 - métricas

![Histórico de métricas da ResNet18](../results/transfer_frozen_modelo/transfer_metrics_history.png)

### DenseNet121 - perda

![Histórico de perda da DenseNet121](../results/transfer_densenet121_modelo/transfer_loss_history.png)

### DenseNet121 - métricas

![Histórico de métricas da DenseNet121](../results/transfer_densenet121_modelo/transfer_metrics_history.png)

## Análise resumida

A ResNet18 com Transfer Learning apresentou o melhor desempenho geral entre os modelos avaliados. Ela superou a CNN simples e a DenseNet121 em acurácia, precisão, recall e F1-score no conjunto de teste.

A DenseNet121 foi mantida como experimento adicional. Apesar de ser uma arquitetura forte e comum em aplicações de imagens médicas, neste experimento ela não superou a ResNet18. Isso reforça a escolha da ResNet18 como modelo final da Pessoa 2.

O recall é uma métrica especialmente relevante neste problema, pois falsos negativos de pneumonia são indesejáveis. A ResNet18 obteve o maior recall entre os modelos comparados.

## Células de validação dos arquivos

As células abaixo são opcionais. Elas carregam os arquivos JSON e PNG usados nas seções anteriores, servindo para conferir se os resultados continuam consistentes no ambiente local.

In [ ]:
from pathlib import Path
import json

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RESULTS_DIR = PROJECT_ROOT / "results"

paths = {
    "simple_cnn_metrics": RESULTS_DIR / "notebook_modelo" / "simple_cnn_metrics.json",
    "resnet_metrics": RESULTS_DIR / "transfer_frozen_modelo" / "transfer_metrics.json",
    "densenet_metrics": RESULTS_DIR / "transfer_densenet121_modelo" / "transfer_metrics.json",
}

missing = [str(path) for path in paths.values() if not path.exists()]
if missing:
    raise FileNotFoundError("Arquivos ausentes:\n" + "\n".join(missing))

print(f"Raiz do projeto: {PROJECT_ROOT}")

In [ ]:
metric_files = [
    paths["simple_cnn_metrics"],
    paths["resnet_metrics"],
    paths["densenet_metrics"],
]

rows = []
for metric_file in metric_files:
    with metric_file.open("r", encoding="utf-8") as file:
        rows.append(json.load(file))

metrics_df = pd.DataFrame(rows)
metrics_df = metrics_df[["model", "accuracy", "precision", "recall", "f1_score"]]
metrics_df = metrics_df.rename(
    columns={
        "model": "Modelo",
        "accuracy": "Acurácia",
        "precision": "Precisão",
        "recall": "Recall",
        "f1_score": "F1-score",
    }
)

metrics_df.style.format({
    "Acurácia": "{:.4f}",
    "Precisão": "{:.4f}",
    "Recall": "{:.4f}",
    "F1-score": "{:.4f}",
})

In [ ]:
confusion_matrices = {
    "CNN simples": RESULTS_DIR / "notebook_modelo" / "simple_cnn_confusion_matrix.png",
    "ResNet18 Transfer Learning": RESULTS_DIR / "transfer_frozen_modelo" / "transfer_confusion_matrix.png",
    "DenseNet121 Transfer Learning": RESULTS_DIR / "transfer_densenet121_modelo" / "transfer_confusion_matrix.png",
}

for title, image_path in confusion_matrices.items():
    if image_path.exists():
        print(title)
        image = mpimg.imread(image_path)
        plt.figure(figsize=(6, 5))
        plt.imshow(image)
        plt.axis("off")
        plt.show()
    else:
        print(f"Arquivo não encontrado: {image_path}")

## Comandos usados

Treino da ResNet18 final:

```bash
python -m src.train_transfer --data-dir data/chest_xray --output-dir results/transfer_frozen_modelo --model resnet18 --epochs 10 --batch-size 32 --lr 0.001 --freeze-backbone
```

Experimento adicional com DenseNet121:

```bash
python -m src.train_transfer --data-dir data/chest_xray --output-dir results/transfer_densenet121_modelo --model densenet121 --epochs 10 --batch-size 32 --lr 0.001 --freeze-backbone
```

Tabela comparativa:

```bash
python -m src.compare_results --simple-cnn results/notebook_modelo/simple_cnn_metrics.json --transfer results/transfer_frozen_modelo/transfer_metrics.json --extra-metrics results/transfer_densenet121_modelo/transfer_metrics.json
```

## Conclusão da Pessoa 2

A ResNet18 com Transfer Learning foi escolhida como modelo final da Pessoa 2. Ela apresentou melhor desempenho no teste e melhor equilíbrio entre as métricas avaliadas. A DenseNet121 foi testada como alternativa, mas ficou abaixo da ResNet18 neste conjunto de dados.

A principal limitação metodológica observada foi o tamanho reduzido do conjunto de validação original. Por isso, a discussão dos resultados deve priorizar as métricas no conjunto de teste, que possui mais imagens e fornece uma avaliação mais estável.